In [ ]:
# !ls /kaggle/input/segmentation-models-pytorch

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
# !pip install --no-deps /kaggle/input/segmentation-models-pytorch/segmentation_models_pytorch-0.5.1.dev0-py3-none-any.whl

In [ ]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
import cv2
import numpy as np
import pandas as pd
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from glob import glob
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.amp import autocast, GradScaler
from matplotlib import pyplot as plt
from collections import OrderedDict
from sklearn.model_selection import KFold
from torch.utils.checkpoint import checkpoint
import timm
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_encoder
import torch.backends.cudnn as cudnn
# from accelerate import Accelerator

cudnn.benchmark = True # Thêm dòng này vào đầu script

In [ ]:
print("timm version:", timm.__version__)
print("smp version:", smp.__version__)
print("Encoders:", list(smp.encoders.encoders.keys())[:10])

In [ ]:
print(torch.__version__)
print(torch.version.cuda)

----- Load data -----

In [ ]:
!cp -r /kaggle/input/dataset/dull_razor_test/kaggle/working/dull_razor_cache_test /kaggle/working/
!cp -r /kaggle/input/dataset/dull_razor_train/kaggle/working/dull_razor_cache /kaggle/working/

# !cp -r /kaggle/input/hyper-parameter/mean_std.json /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold0_best.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold0_checkpoint.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold0_posweight.json /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold1_best.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold1_checkpoint.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold1_posweight.json /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold2_best.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold2_checkpoint.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold2_posweight.json /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold3_best.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold3_checkpoint.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold3_posweight.json /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold4_best.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold4_checkpoint.pth /kaggle/working/
# # !cp -r /kaggle/input/hyper-parameter/fold4_posweight.json /kaggle/working/

In [ ]:
# !rm -rf /kaggle/working/*

In [ ]:
# Lấy đường dẫn data
def load_data_paths(base_path):
    train_imgs  = sorted(glob(os.path.join(base_path, "Train", "**", "Image", "*.jpg")))
    train_masks = sorted(glob(os.path.join(base_path, "Train", "**", "Mask", "*.png")))
    test_imgs   = sorted(glob(os.path.join(base_path, "Test", "**", "Image", "*.jpg")))

    return train_imgs, train_masks, test_imgs

In [ ]:
ROOT_PATH = '/kaggle/input/warm-up-program-ai-vietnam-skin-segmentation'

# check root_path
for root, dirs, files in os.walk(ROOT_PATH):
    print(root, len(files))
    break

train_imgs, train_masks, test_imgs = load_data_paths(ROOT_PATH)

print(f'Size train dataset images: {len(train_imgs)}')
print(f'Size train dataset masks: {len(train_masks)}')
print(f'Size test dataset images: {len(test_imgs)}')

In [ ]:
# vissualize check sample
img = Image.open(train_imgs[0])
mask = Image.open(train_masks[0])

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Image")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Mask")
plt.show()

----- Preprocessing + Augmentation -----

In [ ]:
# ----------------- Hàm tiền xử lý -----------------
def remove_hair(image, small_kernel=9, large_kernel=15, radius=3, 
                use_otsu=True, save_mask=False, debug=False):
    """
    Hair removal cho ảnh ISIC 2018 (improved DullRazor).
    - Multi-scale black-hat
    - Adaptive/Otsu threshold
    - Connected component filter để giữ lại hair thật
    - Inpainting
    
    Args:
        image (np.ndarray): Ảnh BGR đầu vào
        small_kernel (int): kích thước kernel nhỏ (mặc định 9)
        large_kernel (int): kích thước kernel lớn (mặc định 15)
        radius (int): bán kính inpainting (mặc định 3)
        use_otsu (bool): nếu True thì dùng Otsu threshold, False thì dùng ngưỡng cố định
        save_mask (bool): nếu True thì trả thêm mask
        debug (bool): nếu True thì in log
    
    Returns:
        np.ndarray: Ảnh đã loại bỏ hair
        (np.ndarray): Mask hair (nếu save_mask=True)
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # ----- Multi-scale Black-hat -----
    k_small = cv2.getStructuringElement(cv2.MORPH_RECT, (small_kernel, small_kernel))
    k_large = cv2.getStructuringElement(cv2.MORPH_RECT, (large_kernel, large_kernel))
    
    bh_small = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k_small)
    bh_large = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k_large)
    
    # ----- Adaptive Threshold -----
    if use_otsu:
        _, mask_small = cv2.threshold(bh_small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        _, mask_large = cv2.threshold(bh_large, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, mask_small = cv2.threshold(bh_small, 10, 255, cv2.THRESH_BINARY)
        _, mask_large = cv2.threshold(bh_large, 10, 255, cv2.THRESH_BINARY)
    
    # Union mask
    hair_mask = cv2.bitwise_or(mask_small, mask_large)
    
    # ----- Connected Components Filtering -----
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(hair_mask, connectivity=8)
    filtered_mask = np.zeros_like(hair_mask)
    
    for i in range(1, num_labels):  # bỏ background
        x, y, w, h, area = stats[i]
        aspect_ratio = max(w, h) / (min(w, h) + 1e-5)
        
        # Hair: mảnh & dài → area nhỏ, aspect_ratio cao
        if 10 < area < 2000 and aspect_ratio > 2.5:
            filtered_mask[labels == i] = 255
    
    if debug:
        print(f"Detected components: {num_labels-1}, Kept: {np.sum(filtered_mask>0)} px")
    
    # ----- Inpainting -----
    result = cv2.inpaint(image, filtered_mask, radius, cv2.INPAINT_TELEA)
    
    if save_mask:
        return result, filtered_mask
    return result

In [ ]:
# ----------------- Tính Mean/Std -----------------
def get_mean_std(image_paths, save_path="mean_std.json"):
    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            stats = json.load(f)
        return np.array(stats["mean"]), np.array(stats["std"])

    n_images = 0
    mean_sum = np.zeros(3)
    sq_mean_sum = np.zeros(3)

    for path in tqdm(image_paths, desc="Computing mean/std"):
        img = cv2.imread(path, cv2.IMREAD_COLOR).astype(np.float32) / 255.0
        mean_sum += img.mean(axis=(0, 1))
        sq_mean_sum += (img ** 2).mean(axis=(0, 1))
        n_images += 1

    mean = mean_sum / n_images
    mean_of_sq = sq_mean_sum / n_images
    std = np.sqrt(mean_of_sq - mean ** 2)

    stats = {"mean": mean.tolist(), "std": std.tolist()}
    with open(save_path, "w") as f:
        json.dump(stats, f, indent=4)

    return mean, std

In [ ]:
# ! rm -rf /kaggle/working/*

In [ ]:
print(os.listdir('/kaggle/working'))

----------------- Hàm xử lý và tăng cường dữ liệu -----------------

In [ ]:
def process_and_cache(image_paths, save_dir, process_func, desc, save_mask=False):
    """
    Thực hiện tiền xử lý và lưu các ảnh đã xử lý vào một thư mục mới.
    Nếu save_mask=True thì lưu cả mask cùng ảnh.
    """
    processed_paths_file = os.path.join(save_dir, 'processed_paths.json')
    os.makedirs(save_dir, exist_ok=True)
    
    if os.path.exists(processed_paths_file):
        with open(processed_paths_file, 'r') as f:
            processed_paths = json.load(f)
    else:
        processed_paths = []
    
    newly_processed_paths = []
    
    for path in tqdm(image_paths, desc=desc):
        base_name = os.path.basename(path)
        new_path = os.path.join(save_dir, base_name)
        
        if os.path.exists(new_path):
            newly_processed_paths.append(new_path)
            continue
            
        img = cv2.imread(path)
        if img is not None:
            if save_mask:
                processed_img, mask = process_func(img, save_mask=True)
                cv2.imwrite(new_path, processed_img)
                cv2.imwrite(new_path.replace(".jpg", "_mask.png"), mask)
            else:
                processed_img = process_func(img)
                cv2.imwrite(new_path, processed_img)
            
            newly_processed_paths.append(new_path)
    
    with open(processed_paths_file, 'w') as f:
        json.dump(newly_processed_paths, f, indent=4)

    return newly_processed_paths

In [ ]:
def get_image_paths(list_of_paths):
    valid_image_paths = []
    for path in list_of_paths:
        if isinstance(path, str) and os.path.isfile(path) and \
           path.lower().endswith(('.jpg', '.jpeg', '.png')):
            valid_image_paths.append(path)
        # Bỏ qua các đường dẫn không hợp lệ hoặc không phải là ảnh
        # Bạn có thể thêm print(f"Bỏ qua đường dẫn không hợp lệ: {path}") nếu muốn gỡ lỗi.
    return valid_image_paths

In [ ]:
# 1. Tiền xử lý Dull Razor và lưu lại (tập train)
dull_razor_train_paths = process_and_cache(
    train_imgs, 
    './dull_razor_cache', 
    remove_hair, 
    'Applying Dull Razor'
)
print(f"Number of images after Dull Razor processing: {len(dull_razor_train_paths)}")

In [ ]:
# Tiền xử lý Dull Razor và lưu lại (tập test)
dull_razor_test_paths = process_and_cache(
    test_imgs, 
    './dull_razor_cache_test', 
    remove_hair, 
    'Applying Dull Razor Test'
)
print(f"Number of images after Dull Razor processing: {len(dull_razor_test_paths)}")

In [ ]:
img_train_paths = get_image_paths(dull_razor_train_paths)
img_test_paths = get_image_paths(dull_razor_test_paths)
print(f"Number of images img_train_paths: {len(img_train_paths)}")
print(f"Number of images img_test_paths: {len(img_test_paths)}")

In [ ]:
# # 🔹 Tính/lấy mean-std
# mean, std = get_mean_std(img_train_paths, save_path="mean_std.json")
# print(mean, std)

In [ ]:
mean = [0.485, 0.456, 0.406]      # imagenet_mean
std  = [0.229, 0.224, 0.225]      # imagenet_std

In [ ]:
img_size = (512, 512)

train_transform = A.Compose([
    A.Resize(*img_size),

    # 🔹 Biến đổi hình học (an toàn)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_REFLECT_101),

    # 🔹 Biến đổi quang học & màu sắc
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.5),
    A.CLAHE(p=0.3),   # cải thiện contrast vùng lesion
    A.RandomGamma(p=0.3),

    # 🔹 Nhiễu nhẹ
    A.OneOf([
        A.GaussNoise(p=1),
        A.ISONoise(p=1),
    ], p=0.2),

    # 🔹 Blur nhẹ (không quá mạnh để giữ boundary lesion)
    A.GaussianBlur(blur_limit=(3, 3), p=0.2),

    # 🔹 Chuẩn hóa
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(*img_size),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

In [ ]:
# !zip -r /kaggle/working/dull_razor_train.zip /kaggle/working/dull_razor_cache
# !zip -r /kaggle/working/dull_razor_test.zip /kaggle/working/dull_razor_cache_test

----- Create dataset & dataloader -----

In [ ]:
# ----- SegmentationDataset -----
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Đọc ảnh trực tiếp ở dạng màu (3 kênh)
        image = cv2.imread(self.image_paths[idx], cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Albumentations expects RGB
        mask = None

        if self.mask_paths is not None:
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            # Chia mask cho 255 để chuẩn hóa về khoảng [0, 1]
            mask = mask.astype(np.float32) / 255.0

        if self.transform is not None:
            if mask is not None:
                augmented = self.transform(image=image, mask=mask)
                image = augmented["image"]
                # đảm bảo mask là float
                mask = augmented["mask"].float()
            else:
                augmented = self.transform(image=image)
                image = augmented["image"]

        return (image, mask) if mask is not None else image

In [ ]:
# # ----- Create dataset & dataloader -----
# train_imgs_split, val_imgs, train_masks_split, val_masks = train_test_split(
#     dull_razor_train_paths, train_masks, test_size=0.2, random_state=42
# )

# train_dataset = SegmentationDataset(train_imgs_split, train_masks_split, transform=train_transform)
# val_dataset   = SegmentationDataset(val_imgs, val_masks, transform=val_transform)

# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
# val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

----- calculate_pos_weight (mức độ mất cân bằng lớp) -----

In [ ]:
class PosWeightDataset(Dataset):
    """
    Một class Dataset đơn giản chỉ để tính toán pos_weight.
    Chỉ đọc mask và resize chúng để có cùng kích thước.
    """
    def __init__(self, mask_paths, img_size=(512, 512)):
        self.mask_paths = mask_paths
        self.img_size = img_size

    def __len__(self):
        return len(self.mask_paths)

    def __getitem__(self, idx):
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        # Resize mask để đảm bảo tất cả có cùng kích thước
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
        # Thay đổi từ numpy array sang torch tensor
        mask = mask.astype(np.float32) / 255.0 
        return torch.from_numpy(mask)

In [ ]:
# ----- calculate_pos_weight function -----
def calculate_pos_weight(loader):
    """
    Tính toán pos_weight (tỷ lệ giữa số pixel background và forgeground) 
    từ tập dữ liệu huấn luyện.
    """
    total_pos_pixels = 0
    total_neg_pixels = 0
    
    for masks in tqdm(loader, desc="Tính pos_weight"):
        total_pos_pixels += torch.sum(masks > 0).item()
        total_neg_pixels += torch.sum(masks == 0).item()
        
    if total_pos_pixels == 0:
        return 1.0 # Trả về 1 nếu không có pixel dương tính, tránh chia cho 0
    
    return total_neg_pixels / total_pos_pixels

In [ ]:
def get_pos_weight(loader, fold):
    """
    Tải pos_weight từ file nếu tồn tại, ngược lại tính toán và lưu lại.
    """
    save_path = f"fold{fold}_posweight.json"
    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            pos_weight_val = json.load(f)["pos_weight"]
            print(f"Đã tìm thấy file '{save_path}'. Tải giá trị pos_weight: {pos_weight_val:.2f}")
            return pos_weight_val
    else:
        print(f"Không tìm thấy file {save_path}, đang tiến hành tính toán...")
        pos_weight_val = calculate_pos_weight(loader)
        with open(save_path, "w") as f:
            json.dump({"pos_weight": pos_weight_val}, f)
        print(f"Đã tính toán và lưu giá trị pos_weight: {pos_weight_val:.2f} vào file '{save_path}'")
        return pos_weight_val

----------------- Xây dựng Model BCDU-Net -----------------

In [ ]:
# ----------------- Attention blocks (SE + spatial) -----------------
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=True),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x)

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        # x: (B,C,H,W)
        avg = torch.mean(x, dim=1, keepdim=True)
        mx  = torch.max(x, dim=1, keepdim=True)[0]
        cat = torch.cat([avg, mx], dim=1)
        return x * self.sig(self.conv(cat))

In [ ]:
# ----------------- Residual conv block -----------------
class ResConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.proj  = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        res = self.proj(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.relu(x + res)

In [ ]:
# ----------------------
# ConvLSTM cell + wrapper
# ----------------------
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3, bias=True):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                              kernel_size=kernel_size, padding=padding, bias=bias)

    def forward(self, x, h_cur, c_cur):
        # x, h_cur, c_cur: (B, C, H, W)
        combined = torch.cat([x, h_cur], dim=1)
        conv_output = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.chunk(conv_output, 4, dim=1)

        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

In [ ]:
# ----------------- 4-direction Bi-ConvLSTM -----------------
class BiConvLSTM4Dir(nn.Module):
    """
    4-direction BiConvLSTM:
      - forward/back on width (left->right, right->left)
      - forward/back on height (top->bottom, bottom->top)
    Input: (B, C, H, W)
    Output: (B, 4*hid, H, W)  (concatenate hf_w, hb_w, hf_h, hb_h)
    Options:
      - use_checkpoint: use torch.checkpoint on per-step conv to save memory (slower)
    """
    def __init__(self, in_ch, hid_ch=128, kernel_size=3, use_checkpoint=False):
        super().__init__()
        self.hid = hid_ch
        self.fw_w = ConvLSTMCell(in_ch, hid_ch, kernel_size)
        self.bw_w = ConvLSTMCell(in_ch, hid_ch, kernel_size)
        self.fw_h = ConvLSTMCell(in_ch, hid_ch, kernel_size)
        self.bw_h = ConvLSTMCell(in_ch, hid_ch, kernel_size)
        self.use_checkpoint = use_checkpoint

    def _step_fw(self, cell, xt, h, c):
        if self.use_checkpoint:
            # wrap cell step to be checkpointable: need function with tensors as inputs
            def run(x, hh, cc):
                return cell(x, hh, cc)
            h, c = checkpoint(run, xt, h, c)
        else:
            h, c = cell(xt, h, c)
        return h, c

    def forward(self, x):
        B, C, H, W = x.shape
        device = x.device

        # width-wise pass
        hf_w = x.new_zeros(B, self.hid, H, W)
        cf_w = x.new_zeros(B, self.hid, H, W)
        hb_w = x.new_zeros(B, self.hid, H, W)
        cb_w = x.new_zeros(B, self.hid, H, W)

        # forward left->right
        h_fw = x.new_zeros(B, self.hid, H, 1)
        c_fw = x.new_zeros(B, self.hid, H, 1)
        out_fw_list = []
        for t in range(W):
            xt = x[:, :, :, t:t+1]   # (B,C,H,1)
            h_fw, c_fw = self._step_fw(self.fw_w, xt, h_fw, c_fw)
            out_fw_list.append(h_fw)
        # stack -> (B, hid, H, W)
        out_fw = torch.cat(out_fw_list, dim=3)
        hf_w = out_fw

        # backward right->left
        h_bw = x.new_zeros(B, self.hid, H, 1)
        c_bw = x.new_zeros(B, self.hid, H, 1)
        out_bw_list = []
        for t in reversed(range(W)):
            xt = x[:, :, :, t:t+1]
            h_bw, c_bw = self._step_fw(self.bw_w, xt, h_bw, c_bw)
            out_bw_list.append(h_bw)
        out_bw_list.reverse()
        out_bw = torch.cat(out_bw_list, dim=3)
        hb_w = out_bw

        # height-wise pass
        hf_h = x.new_zeros(B, self.hid, H, W)
        cf_h = x.new_zeros(B, self.hid, H, W)
        hb_h = x.new_zeros(B, self.hid, H, W)
        cb_h = x.new_zeros(B, self.hid, H, W)

        # forward top->bottom: iterate over H
        h_fw = x.new_zeros(B, self.hid, 1, W)
        c_fw = x.new_zeros(B, self.hid, 1, W)
        out_fw_list = []
        for t in range(H):
            xt = x[:, :, t:t+1, :]   # (B,C,1,W)
            h_fw, c_fw = self._step_fw(self.fw_h, xt, h_fw, c_fw)
            out_fw_list.append(h_fw)
        out_fw = torch.cat(out_fw_list, dim=2)  # (B,hid,H,W)
        hf_h = out_fw

        # backward bottom->top
        h_bw = x.new_zeros(B, self.hid, 1, W)
        c_bw = x.new_zeros(B, self.hid, 1, W)
        out_bw_list = []
        for t in reversed(range(H)):
            xt = x[:, :, t:t+1, :]
            h_bw, c_bw = self._step_fw(self.bw_h, xt, h_bw, c_bw)
            out_bw_list.append(h_bw)
        out_bw_list.reverse()
        out_bw = torch.cat(out_bw_list, dim=2)
        hb_h = out_bw

        # concat four directions
        out = torch.cat([hf_w, hb_w, hf_h, hb_h], dim=1)  # (B, 4*hid, H, W)
        return out

In [ ]:
# class ConvLSTM(nn.Module):
#     """Simple ConvLSTM: input x_seq shape (B, T, C, H, W) -> returns last hidden state"""
#     def __init__(self, input_dim, hidden_dim, kernel_size=3):
#         super().__init__()
#         self.cell = ConvLSTMCell(input_dim, hidden_dim, kernel_size)

#     def forward(self, x_seq):
#         B, T, C, H, W = x_seq.size()
#         device = x_seq.device
#         h = torch.zeros(B, self.cell.hidden_dim, H, W, device=device)
#         c = torch.zeros(B, self.cell.hidden_dim, H, W, device=device)
#         for t in range(T):
#             h, c = self.cell(x_seq[:, t], h, c)
#         return h  # last hidden state

In [ ]:
# # ----------------------
# # ConvBlock (Conv -> BN -> ReLU)
# # ----------------------
# class ConvBlock(nn.Module):
#     def __init__(self, in_ch, out_ch, bn=True):
#         super().__init__()
#         bias = not bn
#         layers = [
#             nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=bias)
#         ]
#         if bn:
#             layers.append(nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch))
#         layers.append(nn.ReLU(inplace=True))

#         layers.append(nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=bias))
#         if bn:
#             layers.append(nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch))
#         layers.append(nn.ReLU(inplace=True))

#         self.block = nn.Sequential(*layers)

#     def forward(self, x):
#         return self.block(x)

In [ ]:
# ----------------- Decoder (with attention) -----------------
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, use_se=False, use_spatial=False):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ResConvBlock(out_ch + skip_ch, out_ch)
        self.use_se = use_se
        self.use_sp = use_spatial
        if use_se:
            self.se = SEBlock(out_ch)
        if use_spatial:
            self.sp = SpatialAttention()

    def forward(self, x, skip):
        x = self.up(x)
        # ensure size matches (in case of odd sizes)
        if x.size(2) != skip.size(2) or x.size(3) != skip.size(3):
            x = F.interpolate(x, size=(skip.size(2), skip.size(3)), mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        if self.use_se:
            x = self.se(x)
        if self.use_sp:
            x = self.sp(x)
        return x

In [ ]:
# ----------------- Full network -----------------
class ResBCDU_Net_Advanced(nn.Module):
    def __init__(self, backbone='timm-efficientnet-b4', pretrained=True,
                 biconv_hid=128, use_checkpoint=False,
                 use_se=True, use_spatial=True, num_classes=1):
        """
        backbone: name for get_encoder
        biconv_hid: hidden channels for each ConvLSTM cell (final bottleneck channels = 4*biconv_hid)
        use_checkpoint: if True, use checkpoint to save memory in ConvLSTM step
        use_se, use_spatial: attention in decoder
        """
        super().__init__()
        self.encoder = get_encoder(backbone, in_channels=3, depth=5,
                                   weights="imagenet" if pretrained else None)
        enc_chs = self.encoder.out_channels  # [c0, c1, c2, c3, c4, c5]

        self.biconv4 = BiConvLSTM4Dir(in_ch=enc_chs[-1], hid_ch=biconv_hid,
                                      kernel_size=3, use_checkpoint=use_checkpoint)
        bottleneck_ch = 4 * biconv_hid  # since 4 directions

        # decoder blocks: note skip channels are enc_chs[:-1]
        c0, c1, c2, c3, c4, c5 = enc_chs
        self.dec4 = DecoderBlock(bottleneck_ch, c4, 256, use_se=use_se, use_spatial=use_spatial)
        self.dec3 = DecoderBlock(256, c3, 128, use_se=use_se, use_spatial=use_spatial)
        self.dec2 = DecoderBlock(128, c2, 64, use_se=use_se, use_spatial=use_spatial)
        self.dec1 = DecoderBlock(64, c1, 32, use_se=use_se, use_spatial=use_spatial)
        self.dec0 = DecoderBlock(32, c0, 16, use_se=use_se, use_spatial=use_spatial)

        self.out_conv = nn.Sequential(
            nn.Conv2d(16, 8, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(8, num_classes, 1)
        )

    def forward(self, x):
        # encoder returns list [c1..c5]
        feats = self.encoder(x)
        c0, c1, c2, c3, c4, c5 = feats
        # apply 4-direction BiConvLSTM on c5
        c5_bi = self.biconv4(c5)  # (B, 4*hid, H/32, W/32)

        x = self.dec4(c5_bi, c4)
        x = self.dec3(x, c3)
        x = self.dec2(x, c2)
        x = self.dec1(x, c1)
        x = self.dec0(x, c0)
        out = self.out_conv(x)
        return out


In [ ]:
# # ===== Encoder =====
# class Encoder(nn.Module):
#     def __init__(self, in_ch, base_filters, bn=True, dropout=0.2):
#         super().__init__()
        
#         self.enc1 = ConvBlock(in_ch, base_filters, bn=bn)
#         self.pool1 = nn.MaxPool2d(2)

#         self.enc2 = ConvBlock(base_filters, base_filters * 2, bn=bn)
#         self.pool2 = nn.MaxPool2d(2)

#         self.enc3 = ConvBlock(base_filters * 2, base_filters * 4, bn=bn)
#         self.drop3 = nn.Dropout2d(dropout)
#         self.pool3 = nn.MaxPool2d(2)

#     def forward(self, x):
#         c1 = self.enc1(x)
#         p1 = self.pool1(c1)

#         c2 = self.enc2(p1)
#         p2 = self.pool2(c2)

#         c3 = self.enc3(p2)
#         c3_drop = self.drop3(c3)
#         p3 = self.pool3(c3_drop)

#         return c1, c2, c3, c3_drop, p3

In [ ]:
# # ===== Decoder =====
# class Decoder(nn.Module):
#     def __init__(self, base_filters, num_classes, bn=True, dropout=0.2):
#         super().__init__()
#         # Decoder 1 (level 3)
#         self.up6 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
#         self.conv_adapt6 = nn.Conv2d(base_filters * 4, base_filters * 4, kernel_size=1)
#         self.clstm6 = ConvLSTM(base_filters * 4, base_filters * 2)
#         self.dec6 = nn.Sequential(
#             ConvBlock(base_filters * 2, base_filters * 2, bn=bn),
#             nn.Dropout2d(p=dropout)
#         )

#         # Decoder 2 (level 2)
#         self.up7 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
#         self.conv_adapt7 = nn.Conv2d(base_filters * 2, base_filters * 2, kernel_size=1)
#         self.clstm7 = ConvLSTM(base_filters * 2, base_filters)
#         self.dec7 = nn.Sequential(
#             ConvBlock(base_filters, base_filters, bn=bn),
#             nn.Dropout2d(p=dropout)
#         )

#         # Decoder 3 (level 1)
#         self.up8 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
#         self.conv_adapt8 = nn.Conv2d(base_filters, base_filters, kernel_size=1)
#         self.clstm8 = ConvLSTM(base_filters, base_filters // 2)
#         self.dec8 = nn.Sequential(
#             ConvBlock(base_filters // 2, base_filters // 2, bn=bn),
#             nn.Dropout2d(p=dropout)
#         )

#         # Output
#         self.out_conv = nn.Conv2d(base_filters // 2, num_classes, 1)

#     def forward(self, d3, c2, c1, bottleneck):
#         # Decoder 1
#         u6 = self.up6(bottleneck)
#         u6 = self.conv_adapt6(u6)        # align channel trước khi stack
#         m6 = torch.stack([d3, u6], dim=1)
#         m6 = self.clstm6(m6)
#         c6 = self.dec6(m6)

#         # Decoder 2
#         u7 = self.up7(c6)
#         u7 = self.conv_adapt7(u7)
#         m7 = torch.stack([c2, u7], dim=1)
#         m7 = self.clstm7(m7)
#         c7 = self.dec7(m7)

#         # Decoder 3
#         u8 = self.up8(c7)
#         u8 = self.conv_adapt8(u8)
#         m8 = torch.stack([c1, u8], dim=1)
#         m8 = self.clstm8(m8)
#         c8 = self.dec8(m8)

#         return self.out_conv(c8)

In [ ]:
# # ===== BCDU-Net D3 =====
# class BCDUNetD3(nn.Module):
#     def __init__(self, input_channels=1, base_filters=64,
#                  num_classes=1, bn=True):
#         super().__init__()
#         self.encoder = Encoder(input_channels, base_filters, bn=bn)
#         self.center = DenseBlock(in_ch=base_filters*4, growth_rate=base_filters*4, num_layers=3, dropout=0.5)
#         self.decoder = Decoder(base_filters, num_classes, bn=bn)
#         self.num_classes = num_classes

#     def forward(self, x):
#         c1, c2, c3, c3_drop, p3 = self.encoder(x)
#         bottleneck = self.center(p3)
#         logits = self.decoder(c3_drop, c2, c1, bottleneck)

#         # trả về logits, để loss xử lý sigmoid/softmax
#         return logits

----------------- Hàm mất mát và đánh giá -----------------

In [ ]:
# ----- DiceLoss -----
class DiceLoss(nn.Module):
    """
    Dice Loss for binary segmentation.
    This loss function is useful for handling class imbalance.
    """
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        """
        Calculates the Dice Loss.
        Args:
            logits (torch.Tensor): Raw outputs from the model (e.g., [B, 1, H, W]).
            targets (torch.Tensor): Ground truth masks (e.g., [B, H, W] or [B, 1, H, W]).
        Returns:
            torch.Tensor: The mean Dice Loss over the batch.
        """
        # Ensure targets have the same dimensions as logits for broadcasting
        if targets.dim() == 3:
            targets = targets.unsqueeze(1).float()

        # Apply sigmoid to convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Flatten the tensors for easier calculation
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        # Calculate intersection and union
        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)

        # Calculate Dice coefficient
        dice_coeff = (2. * intersection + self.smooth) / (union + self.smooth)

        # Dice Loss is 1 - Dice coefficient
        dice_loss = 1. - dice_coeff.mean()

        return dice_loss

In [ ]:
# ----- FocalLoss -----
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets, pos_weight=None):
        if pos_weight is not None:
            pos_weight = pos_weight.to(inputs.device)
            
        if targets.dim() == 3:
            targets = targets.unsqueeze(1).float()
            
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none', pos_weight=pos_weight)
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        if self.reduction == 'mean':
            return focal.mean()
        elif self.reduction == 'sum':
            return focal.sum()
        return focal

In [ ]:
# # ----- BCEDiceFocalLoss -----
class BCEDiceFocalLoss(nn.Module):
    def __init__(self, use_focal=True, alpha=0.75, focal_alpha=0.25, focal_gamma=2, pos_weight=None):
        super().__init__()
        self.dice = DiceLoss()
        self.use_focal = use_focal
        self.alpha = alpha  # weight for dice vs focal/BCE
        self.pos_weight = pos_weight
        if use_focal:
            self.focal = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        else:
            self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        dice_loss = self.dice(logits, targets)
        if self.use_focal:
            focal_loss = self.focal(logits, targets, pos_weight=self.pos_weight)
            return self.alpha * dice_loss + (1 - self.alpha) * focal_loss
        else:
            bce_loss = self.bce(logits, targets)
            return self.alpha * dice_loss + (1 - self.alpha) * bce_loss

In [ ]:
# ----- dice_score -----
def dice_score(preds, targets, eps=1e-6, threshold=None):
    if targets.ndim == 3:
        targets = targets.unsqueeze(1)
    probs = torch.sigmoid(preds)
    if threshold is not None:  # Hard Dice
        preds_bin = (probs > threshold).float()
        intersection = (preds_bin * targets).sum(dim=(2, 3))
        union = preds_bin.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    else:  # Soft Dice
        intersection = (probs * targets).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean().item()

In [ ]:
# ----- iou_score -----
def iou_score(preds, targets, eps=1e-6, threshold=None):
    if targets.ndim == 3:
        targets = targets.unsqueeze(1)
    probs = torch.sigmoid(preds)
    if threshold is not None:  # Hard IoU
        preds_bin = (probs > threshold).float()
        intersection = (preds_bin * targets).sum(dim=(2, 3))
        union = (preds_bin + targets - preds_bin * targets).sum(dim=(2, 3))
    else:  # Soft IoU
        intersection = (probs * targets).sum(dim=(2, 3))
        union = (probs + targets - probs * targets).sum(dim=(2, 3))

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


In [ ]:
# ----- Evaluate function -----
def evaluate(model, loader, device, criterion=None, threshold=None):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
                
            outputs = model(inputs)

            if targets.dim() == 3:
                targets = targets.unsqueeze(1)

            if outputs.shape[2:] != targets.shape[2:]:
                targets = F.interpolate(targets.float(), size=outputs.shape[2:], mode='nearest')

            if criterion is not None:
                loss = criterion(outputs, targets)
                total_loss += loss.item() * inputs.size(0)

            total_dice += dice_score(outputs, targets, threshold=threshold) * inputs.size(0)
            total_iou += iou_score(outputs, targets, threshold=threshold) * inputs.size(0)

    n = len(loader.dataset)
    mean_loss = total_loss / n if criterion is not None else 0.0
    mean_dice = total_dice / n
    mean_iou = total_iou / n
    return mean_loss, mean_dice, mean_iou

----- Train model -----

In [ ]:
def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None, device="cuda"):
    if os.path.exists(checkpoint_path):
        print(f"Tải checkpoint từ {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=torch.device(device))

        # Tạo một OrderedDict mới và xử lý tiền tố 'module.' một cách an toàn
        new_state_dict = OrderedDict()
        for k, v in checkpoint['model_state_dict'].items():
            name = k[7:] if k.startswith('module.') else k  # Kiểm tra và loại bỏ tiền tố
            new_state_dict[name] = v
        
        # Tải state_dict đã sửa vào mô hình
        # Lưu ý: Nếu mô hình ban đầu không phải là DataParallel, nó sẽ tự động thêm tiền tố 'module.'
        # khi tải một state_dict không có tiền tố này.
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(new_state_dict)
        else:
            model.load_state_dict(new_state_dict)
            
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_dice = checkpoint['best_dice']
        counter = checkpoint['counter']

        return model, optimizer, scheduler, start_epoch, best_dice, counter
    else:
        return model, optimizer, scheduler, 0, 0, 0

In [ ]:
def train_one_epoch(model, train_loader, val_loader, optimizer, device, criterion, scaler):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()

        with autocast(device_type=device):
            logits = model(images)

            # Kiểm tra và điều chỉnh kích thước masks để khớp với logits
            if logits.shape[2:] != masks.shape[2:]:
                masks = F.interpolate(
                    masks.unsqueeze(1).float(), 
                    size=logits.shape[2:], mode='nearest'
                ).squeeze(1)
                
            if masks.dim() == 3:
                masks = masks.unsqueeze(1)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    val_loss, val_dice, val_iou = evaluate(model, val_loader, device, criterion)
    return train_loss, val_loss, val_dice, val_iou
    

In [ ]:
# ----- Train function -----
def train_one_fold(model, pos_weight, train_loader, val_loader, device, fold, epochs=50, lr=1e-5, patience=12):
    # Tính pos_weight trước khi bắt đầu huấn luyện
    # pos_weight = get_pos_weight(train_loader, fold)
    # pos_weight_tensor = torch.tensor([pos_weight], device=device)
    # print(f"Giá trị pos_weight được tính toán: {pos_weight:.3f}")
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    criterion = BCEDiceFocalLoss(pos_weight=pos_weight)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)
    scaler = GradScaler()

    best_dice = 0.0
    counter = 0
    start_epoch = 0

    checkpoint_path = f"fold{fold}_checkpoint.pth"
    
    model, optimizer, scheduler, start_epoch, best_dice, counter = load_checkpoint(
        checkpoint_path, model, optimizer, scheduler, device
    )
    print(f"Resumed fold {fold} from epoch {start_epoch}, best dice {best_dice:.4f}")
    
    train_losses, val_losses, dice_scores, iou_scores = [], [], [], []

    for epoch in range(start_epoch+1, epochs+1):
        train_loss, val_loss, val_dice, val_iou = train_one_epoch(
            model, train_loader, val_loader, optimizer, device, criterion, scaler
        )
        scheduler.step(val_dice)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        dice_scores.append(val_dice)
        iou_scores.append(val_iou)

        print(f"Current learning rate: {optimizer.param_groups[0]['lr']}")
        print(f"Epoch [{epoch}/{epochs}] Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f} | Val Dice: {val_dice:.5f} | Val IoU: {val_iou:.5f}")
            
        # Lưu checkpoint sau mỗi epoch
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_dice,
            'counter': counter
        }
        torch.save(checkpoint, checkpoint_path)
        
        if val_dice > best_dice:
            best_dice = val_dice
            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), f"fold{fold}_best.pth")
            else:
                torch.save(model.state_dict(), f"fold{fold}_best.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered")
                break

    return model, train_losses, val_losses, dice_scores, iou_scores

In [ ]:
def train_model(img_paths, mask_paths, device, epochs=200, lr=1e-4, patience=10):
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(kf.split(img_paths))

    num_gpus = torch.cuda.device_count()
    print(f"Có {num_gpus} GPU khả dụng")

    fold_models, fold_train_losses, fold_val_losses, fold_dice_scores, fold_iou_scores = [], [], [], [], []
    for fold, (tr, val) in enumerate(splits):
        print(f"[Fold {fold}] Training on {device}, Train size: {len(tr)}, Val size: {len(val)}")

        # Tính pos_weight trước khi bắt đầu huấn luyện
        pos_weight_dataset = PosWeightDataset([mask_paths[i] for i in tr])
        pos_weight_loader = DataLoader(pos_weight_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
        pos_weight_value = get_pos_weight(pos_weight_loader, fold)
        with autocast(device_type='cuda', enabled=False):
            pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32)  # CPU
        pos_weight_tensor = pos_weight_tensor.to(device)  # GPU   
        print(f"Giá trị pos_weight được tính toán: {pos_weight_value:.3f}")
        
        train_dataset = SegmentationDataset([img_paths[i] for i in tr], [mask_paths[i] for i in tr], transform=train_transform)
        val_dataset   = SegmentationDataset([img_paths[i] for i in val], [mask_paths[i] for i in val], transform=val_transform)
        
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
    
        model = ResBCDU_Net_Advanced(
            backbone="timm-efficientnet-b0",
            pretrained=True,
            biconv_hid=128,          # hidden channels per direction
            use_checkpoint=False,     # save VRAM at cost of compute
            use_se=True,
            use_spatial=True,
            num_classes=1
        ).to(device)

        # if torch.cuda.device_count() > 1:
        #     model = nn.DataParallel(model)
            
        model, train_losses, val_losses, dice_scores, iou_scores = train_one_fold(
            model, pos_weight_tensor, train_loader, val_loader, device, fold, epochs=epochs, lr=lr, patience=patience
        )
        if os.path.exists(f"fold{fold}_best.pth"):
            if isinstance(model, nn.DataParallel):
                model.module.load_state_dict(torch.load(f"fold{fold}_best.pth"))
            else:
                model.load_state_dict(torch.load(f"fold{fold}_best.pth"))
                
            fold_models.append(f"fold{fold}_best.pth")
            fold_train_losses.append(train_losses)
            fold_val_losses.append(val_losses)
            fold_dice_scores.append(dice_scores)
            fold_iou_scores.append(iou_scores)

    return fold_models, fold_train_losses, fold_val_losses, fold_dice_scores, fold_iou_scores

In [ ]:
# ! rm -r /kaggle/working/state.db

In [ ]:
# import gc

# gc.collect()
# torch.cuda.empty_cache()  

In [ ]:
# ----- Model Execution -----
device = "cuda" if torch.cuda.is_available() else "cpu"

fold_models, fold_train_losses, fold_val_losses, fold_dice_scores, fold_iou_scores = train_model(
    img_train_paths, train_masks, device, epochs=150, lr=1e-4, patience=10
)

In [ ]:
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
# fold_train_losses, fold_val_losses, fold_dice_scores, fold_iou_scores
print(len(fold_train_losses[4]))
print(len(fold_val_losses[4]))
print(len(fold_dice_scores[4]))
print(len(fold_iou_scores[4]))

----- Vissualization -----

In [ ]:
# Plot training curves
def plot_training_curves(train_losses, val_losses, dice_scores, iou_scores):
    epochs = range(1, len(train_losses) + 1)

    plt.style.use("ggplot")
    plt.figure(figsize=(14,5))

    # Loss
    plt.subplot(1,2,1)
    plt.plot(epochs, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train & Validation Loss')
    plt.legend()
    plt.grid(True)

    # Metrics
    plt.subplot(1,2,2)
    plt.plot(epochs, dice_scores, 'g-', label='Dice Score')
    plt.plot(epochs, iou_scores, 'm-', label='IoU Score')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.title('Dice & IoU Scores')
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
for i, _ in enumerate(fold_train_losses):
    print(f"📈 Fold {i+1}")
    plot_training_curves(
        fold_train_losses[i], 
        fold_val_losses[i], 
        fold_dice_scores[i], 
        fold_iou_scores[i]
    )

In [ ]:
# file_path = "training_log.csv"
# if os.path.exists(file_path):
#     df = pd.read_csv(file_path)

#     train_losses = df['train_loss'].tolist()
#     val_losses = df['val_loss'].tolist()
#     dice_scores = df['val_dice'].tolist()
#     iou_scores = df['val_iou'].tolist()

#     plot_training_curves(train_losses, val_losses, dice_scores, iou_scores)
# else:
#     plot_training_curves(train_losses, val_losses, dice_scores, iou_scores)

In [ ]:
# ----------------- Hàm mã hóa RLE và tạo file nộp bài -----------------
def rle_encode(mask):
    '''
    mask: numpy array, 1 - mask, 0 - background
    returns: run-length encoded string
    '''
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

In [ ]:
def create_submission_file(output_dir, test_paths, submission_file="submission.csv"):
    print("Bắt đầu tạo file nộp bài...")
    submission_list = []
    for path in tqdm(test_paths, desc="Đang mã hóa RLE"):
        filename = os.path.basename(path).replace(".jpg", ".png")
        mask_path = os.path.join(output_dir, filename)
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            print(filename, mask.shape, mask.max(), mask.min())
            rle = rle_encode(mask)
            submission_list.append([filename, rle])

    submission_df = pd.DataFrame(submission_list, columns=['ID', 'Predicted_Mask'])
    submission_df.to_csv(submission_file, index=False)
    print(f"Đã tạo file nộp bài thành công: {submission_file}")

In [ ]:
def load_fold_models(fold_models, device):
    models = []
    for fold_model in fold_models:
        model = ResBCDU_Net_Advanced(
            backbone="timm-efficientnet-b0",
            pretrained=True,
            biconv_hid=128,          # hidden channels per direction
            use_checkpoint=False,     # save VRAM at cost of compute
            use_se=True,
            use_spatial=True,
            num_classes=1
        ).to(device)
        state_dict = torch.load(fold_model, map_location=device)
        model.load_state_dict(state_dict)
        model.eval()
        models.append(model)
        print(f"Loaded {fold_model}")
    return models

In [ ]:
def ensemble_predict(models, image, device):
    preds = []
    with torch.no_grad():
        for model in models:
            out = model(image)                # (B,1,H,W)
            pred = torch.sigmoid(out).cpu().numpy()
            preds.append(pred[0,0])  # đảm bảo (H,W)
    return np.mean(preds, axis=0)

In [ ]:
def post_process_mask(mask, min_area=0):
    """
    mask: numpy array (H,W), nhị phân (0/1)
    """
    mask = mask.astype(np.uint8)
    num_components, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    new_mask = np.zeros_like(mask)
    for i in range(1, num_components):  # bỏ background
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            new_mask[labels == i] = 1
    return new_mask

In [ ]:
def predict_with_tta(models, test_paths, device, transform, 
                     output_dir="predictions", thresholds=[0.35, 0.45, 0.5], min_area=0):
    """
    Ensemble + TTA + Multi-threshold voting cho toàn bộ test set
    """
    print("🔮 Bắt đầu dự đoán test set với Ensemble + TTA + Multi-threshold voting...")
    os.makedirs(output_dir, exist_ok=True)

    # Các phép TTA: (augment_fn, invert_fn)
    tta_transforms = [
        (lambda x: x, lambda x: x),  # gốc
        (lambda x: np.flip(x, axis=1), lambda x: np.flip(x, axis=1)),  # flip H
        (lambda x: np.flip(x, axis=0), lambda x: np.flip(x, axis=0)),  # flip V
        (lambda x: np.flip(np.flip(x, axis=0), axis=1), lambda x: np.flip(np.flip(x, axis=0), axis=1)),  # flip HV
    ]

    with torch.no_grad():
        for img_path in tqdm(test_paths, desc="Dự đoán"):
            # Đọc ảnh gốc (RGB)
            img_raw = cv2.imread(img_path, cv2.IMREAD_COLOR)
            img_raw = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)

            tta_preds = []

            for aug, deaug in tta_transforms:
                # Apply TTA (augment ảnh numpy trước khi transform)
                img_aug = aug(img_raw)

                # Chuẩn hóa & sang tensor
                img_tensor = transform(image=img_aug)["image"].unsqueeze(0).to(device)

                # Ensemble predict
                pred = ensemble_predict(models, img_tensor, device)  # (H,W)

                # Inverse TTA
                pred = deaug(pred)
                tta_preds.append(pred)

            # Trung bình kết quả TTA
            final_prob = np.mean(tta_preds, axis=0)

            # Multi-threshold voting
            bin_masks = [(final_prob > th).astype(np.uint8) for th in thresholds]
            voted_mask = np.stack(bin_masks, axis=0).sum(axis=0)
            voted_mask = (voted_mask >= (len(thresholds) // 2 + 1)).astype(np.uint8)

            # Hậu xử lý: bỏ vùng nhỏ hơn min_area
            final_mask = post_process_mask(voted_mask, min_area=min_area)

            # Resize về 512x512 trước khi lưu
            # final_mask = cv2.resize(final_mask, (512, 512), interpolation=cv2.INTER_NEAREST)

            # Lưu mask
            out_path = os.path.join(output_dir, os.path.basename(img_path).replace(".jpg", ".png"))
            cv2.imwrite(out_path, final_mask * 255)

    print(f"✅ Đã lưu toàn bộ kết quả vào: {output_dir}")

In [ ]:
# # Thư mục chứa các file
# working_dir = '/kaggle/working'

# # Danh sách để lưu tên các file .pth
# fold_models = []

# # Lặp qua tất cả các mục trong thư mục
# for fold in range(5):
#     # Kiểm tra xem mục có phải là file và có đuôi .pth không
#         # Thêm tên file vào list
#     fold_models.append(f"fold{fold}_best.pth") # Thêm đường dẫn đầy đủ nếu cần

# print(fold_models)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load các model từ k-fold
models = load_fold_models(fold_models, device)

# Predict test set
predict_with_tta(models, img_test_paths, device, val_transform, output_dir="predictions")
create_submission_file(output_dir="predictions", test_paths=test_imgs, submission_file="prediction.csv")

In [ ]:
import os
import cv2
import glob
import matplotlib.pyplot as plt

def plot_img_mask(submission_folder):
    # Kiểm tra xem thư mục có tồn tại và có chứa file nào không
    if not os.path.exists(submission_folder) or not glob.glob(os.path.join(submission_folder, "*.png")):
        print("Thư mục 'submission' không tồn tại hoặc không chứa file mask nào.")
    else:
        # Lấy đường dẫn của file mask
        first_mask_path = sorted(glob.glob(os.path.join(submission_folder, "*.png")))[6]
        img_path = test_imgs[6]
        
        # Đọc mask bằng OpenCV
        # OpenCV đọc ảnh theo định dạng BGR, nhưng mask chỉ có một kênh màu xám
        img = Image.open(first_mask_path)
        mask = Image.open(img_path).resize((512,512))
        
        # Hiển thị mask
        plt.figure(figsize=(8, 8))
        plt.subplot(1, 2, 1)
        plt.imshow(mask, cmap='gray')
        plt.title(f"Visualizing Mask: {os.path.basename(first_mask_path)}")
        plt.axis('off')
        plt.show()
    
        plt.subplot(1, 2, 2)
        plt.imshow(img)
        plt.title(f"Visualizing Mask: {os.path.basename(img_path)}")
        plt.axis('off')
        plt.show()
    print("Đã hoàn thành việc hiển thị mask.")


In [ ]:
submission_folder = "predictions"
plot_img_mask(submission_folder)

In [ ]:
def visualize_prediction(img_path, mask_gt_path, models, transform, device):
    # đọc ảnh
    img_raw = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    
    # ground truth mask
    mask_gt = cv2.imread(mask_gt_path, cv2.IMREAD_GRAYSCALE)
    
    # preprocess
    img_tensor = transform(image=img_rgb)["image"].unsqueeze(0).to(device)

    # predict
    pred = ensemble_predict(models, img_tensor, device)  # (H,W) xác suất
    pred_bin = (pred > 0.5).astype(np.uint8)  # threshold

    # plot
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    axs[0].imshow(img_rgb)
    axs[0].set_title("Ảnh gốc")
    axs[1].imshow(mask_gt, cmap="gray")
    axs[1].set_title("GT Mask")
    axs[2].imshow(pred, cmap="viridis")
    axs[2].set_title("Pred Prob (0–1)")
    axs[3].imshow(pred_bin, cmap="gray")
    axs[3].set_title("Pred Binary (>0.5)")
    plt.show()

# ví dụ
visualize_prediction("/kaggle/working/dull_razor_cache/ISIC_0000001.jpg", "/kaggle/working/dull_razor_cache_test/ISIC_0012169.jpg", models, val_transform, device)

In [ ]:
# Hàm dự đoán sử dụng Test-Time Augmentation
def predict_with_tta(models, test_paths, device, original_transform, output_dir="submission_tta", threshold=0.5):
    print("Bắt đầu dự đoán trên tập dữ liệu kiểm tra với TTA...")
    os.makedirs(output_dir, exist_ok=True)
    model.eval()

    # Định nghĩa các phép biến đổi TTA và các phép biến đổi ngược tương ứng
    tta_transforms = [
        # (hàm biến đổi, hàm đảo ngược)
        (lambda img: img, lambda mask: mask),  # Gốc
        (lambda img: cv2.flip(img, 1), lambda mask: cv2.flip(mask, 1)), # Lật ngang
        (lambda img: cv2.flip(img, 0), lambda mask: cv2.flip(mask, 0)), # Lật dọc
        (lambda img: cv2.flip(img, -1), lambda mask: cv2.flip(mask, -1)), # Lật ngang và dọc
        (lambda img: cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE), lambda mask: cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)), # Xoay 90 độ
        (lambda img: cv2.rotate(img, cv2.ROTATE_180), lambda mask: cv2.rotate(mask, cv2.ROTATE_180)), # Xoay 180 độ
        (lambda img: cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE), lambda mask: cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)), # Xoay 270 độ
    ]
        
    with torch.no_grad():
        for original_path in tqdm(test_paths, desc="Đang dự đoán với TTA"):
            image_raw = cv2.imread(original_path, cv2.IMREAD_COLOR)
            image_raw = cv2.cvtColor(image_raw, cv2.COLOR_BGR2RGB)
            
            tta_masks = []
            
            for transform, inverse_transform in tta_transforms:
                # Áp dụng TTA
                augmented_image = transform(image_raw)

                # Chuẩn hóa và chuyển đổi sang tensor
                transformed_input = original_transform(image=augmented_image)["image"].unsqueeze(0).to(device)
                
                # Dự đoán
                pred_mask = ensemble_predict(models, img, device)
                
                # Áp dụng phép biến đổi ngược
                inverted_mask = inverse_transform(pred_mask)
                
                tta_masks.append(inverted_mask)

            # Hợp nhất các dự đoán bằng cách tính trung bình
            final_mask = np.mean(tta_masks, axis=0)
            final_mask = (final_mask > threshold).astype(np.uint8) * 255
            
            # Áp dụng hậu xử lý
            processed_mask = post_process_mask(final_mask)
            
            filename = os.path.basename(original_path)
            output_path = os.path.join(output_dir, filename.replace(".jpg", ".png"))
            cv2.imwrite(output_path, processed_mask)
    
    print(f"Đã lưu các mask dự đoán với TTA vào thư mục '{output_dir}'")

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"

# # Load các model từ k-fold
# models = load_fold_models(fold_models, device)

# # Predict test set
# predict_with_tta(models, img_test_paths, device, val_transform, output_dir="predictions_0_3")
# create_submission_file(output_dir="predictions_0_3", test_paths=test_imgs, submission_file="prediction_0_3.csv")

----------------- Dự đoán trên tập dữ liệu kiểm tra (chưa tta) -----------------

In [ ]:
# # ----------------- Hàm dự đoán trên tập dữ liệu kiểm tra -----------------
# def predict(model, test_paths, device, transform, output_dir="submission"):
#     print("Bắt đầu dự đoán trên tập dữ liệu kiểm tra...")
#     os.makedirs(output_dir, exist_ok=True)
#     model.eval()
    
#     test_dataset = SegmentationDataset(test_paths, transform=transform)
#     test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4)

#     with torch.no_grad():
#         for i, images in tqdm(enumerate(test_loader), total=len(test_loader), desc="Đang dự đoán"):
#             images = images.to(device)
#             outputs = model(images)
            
#             # Chuyển đổi logits thành mask nhị phân (0 hoặc 255)
#             preds = torch.sigmoid(outputs)
#             preds = (preds > 0.5).float() * 255
            
#             for j in range(preds.size(0)):
#                 # Lấy tên file gốc
#                 original_path = test_paths[i * test_loader.batch_size + j]
#                 filename = os.path.basename(original_path)
                
#                 # Lưu mask
#                 mask_np = preds[j].squeeze().cpu().numpy().astype(np.uint8)
#                 output_path = os.path.join(output_dir, filename.replace(".jpg", ".png"))
#                 cv2.imwrite(output_path, mask_np)
#     print(f"Đã lưu các mask dự đoán vào thư mục '{output_dir}'")

In [ ]:
# ! rm -r /kaggle/working/submission.csv submission_tta submission_tta_best_thredshold submission_tta_no_thredshold

In [ ]:
# # Dự đoán trên tập dữ liệu kiểm tra sau khi huấn luyện
# predict(model, dull_razor_test_paths, device, val_transform)

In [ ]:
# # Tạo file nộp bài sau khi dự đoán
# create_submission_file(output_dir="submission", test_paths=test_imgs)

In [ ]:
# # Đường dẫn đến thư mục chứa các mask dự đoán
# submission_folder = "submission"
# plot_img_mask(submission_folder)

In [ ]:
print(os.listdir('/kaggle/working'))

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta) -----------------

In [ ]:
# # ----------------- Hàm dự đoán trên tập dữ liệu kiểm tra -----------------
# def post_process_mask(mask, kernel_scale=128, area_ratio=0.002):
#     """
#     Hậu xử lý mask cho dữ liệu ISIC 2018.
#     - kernel_scale: điều chỉnh kernel dựa trên kích thước ảnh (mặc định: 1/128 chiều ảnh).
#     - area_ratio: tỷ lệ diện tích tối thiểu để giữ lại vùng (mặc định: 0.2% diện tích ảnh).
#     """
#     # Đảm bảo mask là nhị phân (0, 255)
#     mask = (mask > 127).astype(np.uint8) * 255

#     h, w = mask.shape[:2]

#     # Kernel adaptive theo kích thước ảnh
#     kernel_size = max(3, (min(h, w) // kernel_scale) | 1)  # luôn lẻ
#     kernel = np.ones((kernel_size, kernel_size), np.uint8)

#     # Morphological closing để lấp lỗ nhỏ
#     mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

#     # Ngưỡng diện tích nhỏ nhất (theo % diện tích ảnh)
#     min_area = h * w * area_ratio

#     # Tìm và loại bỏ các vùng nhỏ hơn min_area
#     contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     for contour in contours:
#         if cv2.contourArea(contour) < min_area:
#             cv2.drawContours(mask, [contour], -1, 0, -1)

#     # Đảm bảo nhị phân sau xử lý
#     mask = (mask > 127).astype(np.uint8) * 255
#     return mask

In [ ]:
# # Hàm tìm ngưỡng tối ưu
# def find_optimal_threshold(model, val_loader, device, num_thresholds=100):
#     """
#     Tìm ngưỡng tối ưu trên tập validation một cách tiết kiệm bộ nhớ.
#     Sử dụng cache để lưu lại kết quả.
#     """
#     cache_file = "optimal_threshold_results.json"
#     if os.path.exists(cache_file):
#         with open(cache_file, "r") as f:
#             results = json.load(f)
#             best_threshold = results["optimal_threshold"]
#             best_dice = results["dice_score"]
#             print(f"Đã tìm thấy file '{cache_file}'. Tải kết quả từ lần chạy trước.")
#             print(f"Ngưỡng tối ưu đã lưu: {best_threshold:.3f}, Dice Score tương ứng: {best_dice:.5f}")
#             return best_threshold, best_dice

#     print("Bắt đầu tìm ngưỡng tối ưu...")
#     model.eval()
#     thresholds = np.linspace(0.01, 0.99, num_thresholds)
#     best_dice = 0.0
#     best_threshold = 0.5
    
#     # Lưu kết quả Dice Score của từng ngưỡng vào một dictionary
#     threshold_results = {threshold: 0.0 for threshold in thresholds}
    
#     with torch.no_grad():
#         for inputs, targets in tqdm(val_loader, desc="Đang tính toán và tìm ngưỡng"):
#             inputs, targets = inputs.to(device), targets.to(device)
#             outputs = model(inputs)
#             probs = torch.sigmoid(outputs)

#             if targets.dim() == 3:
#                 targets = targets.unsqueeze(1)

#             for threshold in thresholds:
#                 preds_bin = (probs > threshold).float()
                
#                 # Tính toán Dice Score cho batch hiện tại
#                 intersection = (preds_bin * targets).sum(dim=(2, 3))
#                 union = preds_bin.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
#                 dice_per_image = (2. * intersection + 1e-6) / (union + 1e-6)
                
#                 # Tích lũy tổng Dice Score cho ngưỡng này
#                 threshold_results[threshold] += dice_per_image.sum().item()

#     # Tính Dice Score trung bình cho từng ngưỡng
#     n_images = len(val_loader.dataset)
#     for threshold, total_dice in threshold_results.items():
#         avg_dice = total_dice / n_images
#         print(f"Ngưỡng: {threshold:.3f}, Dice Score: {avg_dice:.5f}")

#         if avg_dice > best_dice:
#             best_dice = avg_dice
#             best_threshold = threshold
            
#     print(f"\nNgưỡng tối ưu tìm được trên tập validation là: {best_threshold:.3f}")
#     print(f"Dice Score tương ứng là: {best_dice:.5f}")
    
#     # Lưu kết quả vào file
#     with open(cache_file, "w") as f:
#         json.dump({"optimal_threshold": best_threshold, "dice_score": best_dice}, f, indent=4)
        
#     return best_threshold, best_dice

In [ ]:
# def predict(model, test_paths, device, transform, output_dir="submission"):
#     print("Bắt đầu dự đoán trên tập dữ liệu kiểm tra...")
#     os.makedirs(output_dir, exist_ok=True)
#     model.eval()
    
#     test_dataset = SegmentationDataset(test_paths, transform=transform)
#     test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4)

#     with torch.no_grad():
#         for i, images in tqdm(enumerate(test_loader), total=len(test_loader), desc="Đang dự đoán"):
#             images = images.to(device)
#             outputs = model(images)
            
#             preds = torch.sigmoid(outputs)
            
#             for j in range(preds.size(0)):
#                 original_path = test_paths[i * test_loader.batch_size + j]
#                 filename = os.path.basename(original_path)
                
#                 mask_np = preds[j].squeeze().cpu().numpy()
#                 mask_np = (mask_np > 0.5).astype(np.uint8) * 255
                
#                 # Áp dụng hậu xử lý
#                 processed_mask = post_process_mask(mask_np)
                
#                 output_path = os.path.join(output_dir, filename.replace(".jpg", ".png"))
#                 cv2.imwrite(output_path, processed_mask)
#     print(f"Đã lưu các mask dự đoán vào thư mục '{output_dir}'")

In [ ]:
# # Hàm dự đoán sử dụng Test-Time Augmentation
# def predict_with_tta(models, test_paths, device, original_transform, output_dir="submission_tta", threshold=0.5):
#     print("Bắt đầu dự đoán trên tập dữ liệu kiểm tra với TTA...")
#     os.makedirs(output_dir, exist_ok=True)
#     model.eval()

#     # Định nghĩa các phép biến đổi TTA và các phép biến đổi ngược tương ứng
#     tta_transforms = [
#         # (hàm biến đổi, hàm đảo ngược)
#         (lambda img: img, lambda mask: mask),  # Gốc
#         (lambda img: cv2.flip(img, 1), lambda mask: cv2.flip(mask, 1)), # Lật ngang
#         (lambda img: cv2.flip(img, 0), lambda mask: cv2.flip(mask, 0)), # Lật dọc
#         (lambda img: cv2.flip(img, -1), lambda mask: cv2.flip(mask, -1)), # Lật ngang và dọc
#         (lambda img: cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE), lambda mask: cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)), # Xoay 90 độ
#         (lambda img: cv2.rotate(img, cv2.ROTATE_180), lambda mask: cv2.rotate(mask, cv2.ROTATE_180)), # Xoay 180 độ
#         (lambda img: cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE), lambda mask: cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)), # Xoay 270 độ
#     ]
        
#     with torch.no_grad():
#         for original_path in tqdm(test_paths, desc="Đang dự đoán với TTA"):
#             image_raw = cv2.imread(original_path, cv2.IMREAD_COLOR)
#             image_raw = cv2.cvtColor(image_raw, cv2.COLOR_BGR2RGB)
            
#             tta_masks = []
            
#             for transform, inverse_transform in tta_transforms:
#                 # Áp dụng TTA
#                 augmented_image = transform(image_raw)

#                 # Chuẩn hóa và chuyển đổi sang tensor
#                 transformed_input = original_transform(image=augmented_image)["image"].unsqueeze(0).to(device)
                
#                 # Dự đoán
#                 pred_mask = ensemble_predict(models, img, device)
                
#                 # Áp dụng phép biến đổi ngược
#                 inverted_mask = inverse_transform(pred_mask)
                
#                 tta_masks.append(inverted_mask)

#             # Hợp nhất các dự đoán bằng cách tính trung bình
#             final_mask = np.mean(tta_masks, axis=0)
#             final_mask = (final_mask > threshold).astype(np.uint8) * 255
            
#             # Áp dụng hậu xử lý
#             processed_mask = post_process_mask(final_mask)
            
#             filename = os.path.basename(original_path)
#             output_path = os.path.join(output_dir, filename.replace(".jpg", ".png"))
#             cv2.imwrite(output_path, processed_mask)
    
#     print(f"Đã lưu các mask dự đoán với TTA vào thư mục '{output_dir}'")

In [ ]:
# # Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
# predict_with_tta(models, dull_razor_test_paths, device, val_transform, output_dir="submission_tta", threshold=0.5)

In [ ]:
# # Tạo file nộp bài sau khi dự đoán
# create_submission_file(output_dir="submission_tta", test_paths=test_imgs, submission_file="submission_tta.csv")

In [ ]:
# # Đường dẫn đến thư mục chứa các mask dự đoán
# submission_folder = "submission_tta"
# plot_img_mask(submission_folder)

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta - best thredshold) -----------------

In [ ]:
# best_threshold, best_dice_val = find_optimal_threshold(model, val_loader, device)

In [ ]:
# # Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
# predict_with_tta(model, dull_razor_test_paths, device, val_transform, output_dir="submission_tta_best_thredshold", threshold=best_threshold)

In [ ]:
# # Tạo file nộp bài sau khi dự đoán
# create_submission_file(output_dir="submission_tta_best_thredshold", test_paths=test_imgs, submission_file="submission_tta_best_thredshold.csv")

In [ ]:
# # Đường dẫn đến thư mục chứa các mask dự đoán
# submission_folder = "submission_tta_best_thredshold"
# plot_img_mask(submission_folder)

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta - 0.4 thredshold) -----------------

In [ ]:
# # Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
# predict_with_tta(model, dull_razor_test_paths, device, val_transform, output_dir="submission_tta_0_4_thredshold", threshold=0.4)

In [ ]:
# # Tạo file nộp bài sau khi dự đoán
# create_submission_file(output_dir="submission_tta_0_4_thredshold", test_paths=test_imgs, submission_file="submission_tta_0_4_thredshold.csv")

In [ ]:
# # Đường dẫn đến thư mục chứa các mask dự đoán
# submission_folder = "submission_tta_0_4_thredshold"
# plot_img_mask(submission_folder)

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta - 0.2 thredshold) -----------------

In [ ]:
# # Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
# predict_with_tta(model, dull_razor_test_paths, device, val_transform, output_dir="submission_tta_0_2_thredshold", threshold=0.2)

In [ ]:
# # Tạo file nộp bài sau khi dự đoán
# create_submission_file(output_dir="submission_tta_0_2_thredshold", test_paths=test_imgs, submission_file="ubmission_tta_0_2_thredshold.csv")

In [ ]:
# # Đường dẫn đến thư mục chứa các mask dự đoán
# submission_folder = "submission_tta_0_2_thredshold"
# plot_img_mask(submission_folder)

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta - 0.1 thredshold) -----------------

In [ ]:
# # Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
# predict_with_tta(model, dull_razor_test_paths, device, val_transform, output_dir="submission_tta_0_1_thredshold", threshold=0.1)

In [ ]:
# # Tạo file nộp bài sau khi dự đoán
# create_submission_file(output_dir="submission_tta_0_1_thredshold", test_paths=test_imgs, submission_file="ubmission_tta_0_1_thredshold.csv")

In [ ]:
# # Đường dẫn đến thư mục chứa các mask dự đoán
# submission_folder = "submission_tta_0_1_thredshold"
# plot_img_mask(submission_folder)

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta - no thredshold) -----------------

In [ ]:
# # Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
# predict_with_tta(model, dull_razor_test_paths, device, val_transform, output_dir="submission_tta_no_thredshold", threshold=0)

In [ ]:
# # Tạo file nộp bài sau khi dự đoán
# create_submission_file(output_dir="submission_tta_no_thredshold", test_paths=test_imgs, submission_file="submission_tta_no_thredshold.csv")

In [ ]:
# # Đường dẫn đến thư mục chứa các mask dự đoán
# submission_folder = "submission_tta_no_thredshold"
# plot_img_mask(submission_folder)